# Phase 4 — Submission 01: Model 4 Peak Capacity Boundary Decoder (Full FT Anchor & KD Teacher)

**Task:** Court Judgment Prediction (CJPE) in Indian Legal Documents (ILDC Benchmark)  
**Task Type:** Document-Level Binary Classification (`0` = Dismissed/Rejected, `1` = Allowed/Accepted)  
**Primary Evaluation Metric:** Macro-F1 Score  
**Target Backbone:** Phase 1 Model 4 Peak Capacity Boundary Decoder (153.9M parameters, $L=14, d=896, H_q=14, H_{kv}=7$, GQA 2:1, SwiGLU $d_{ff}=2688$, RoPE)  
**Pretrained Hub Location:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `model4-capacity-boundary-154m`)  
**Target Hub Destination:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `Phase_4/sub01_model4_full_ft/`)  

---

### Strategic Architectural Role & Teacher Calibration
This notebook serves two vital functions in the Phase 4 suite:
1. **Performance Upper-Bound Anchor:** Model 4 possesses the highest parameter capacity (153.9M) and widest hidden dimension ($d=896$) among all Phase 1 backbones. Full parameter fine-tuning establishes the empirical ceiling of the model family.
2. **Knowledge Distillation Teacher for Sub 06:** Sub 01 acts as the teacher for Sub 06 (Model 3 Statutory Precision Full-MHA, 109.5M), fulfilling the compulsory capacity reduction requirement ($153.9\text{M} \to 109.5\text{M}$).
3. **Early-Stopping Calibration:** Trained for **2 epochs** with low backbone learning rate ($5.0 \times 10^{-5}$) to prevent overconfidence and logit saturation before exporting soft teacher targets $\sigma(z / T)$ at temperature $T=2.0$ into `sub01_teacher_logits.pt`.
4. **Tail Chunking & Attention Pooling:** Documents are segmented into 1024-token non-overlapping chunks. The last $K=4$ chunks are pooled via learned attention over $\text{Concat}(\text{MeanPool}, \text{LastToken})$ chunk representations ($2 \cdot d = 1792$).
5. **Automated Hub Synchronization:** All model weights, teacher soft logits, Codabench `submission.zip`, metrics, plots, and **the executed notebook itself** are atomically published to Hugging Face Hub in a single commit.


In [ ]:
# Cell 1: Imports, Environment Setup & Upfront Hugging Face Authentication
import os
import sys
import json
import math
import time
import shutil
import zipfile
import random
import getpass
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import PreTrainedTokenizerFast, AutoConfig, AutoModel, get_cosine_schedule_with_warmup
from safetensors.torch import load_file as safetensors_load_file
from huggingface_hub import hf_hub_download, login, get_token, HfApi

# 1. Reproducible Seed Initialization
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Compute Device Configuration with Architecture Compatibility Guard
def get_safe_device(preferred_device: Optional[str] = None) -> torch.device:
    if preferred_device:
        return torch.device(preferred_device)
    if torch.cuda.is_available():
        try:
            t = torch.zeros(1, device="cuda")
            _ = t + 1
            return torch.device("cuda")
        except Exception as e:
            print(f"Notice: CUDA is present but incompatible with current PyTorch build ({e}). Falling back to CPU.")
            return torch.device("cpu")
    return torch.device("cpu")

device = get_safe_device()
print(f"Active Compute Device: {device}")
if device.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

# 3. Repository and Target Configuration
MODEL_REPO = "sarweshbarnwal/CS781-Legal-Models"
TARGET_SUBFOLDER = "Phase_4/sub01_model4_full_ft"
DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
BACKBONE_SUBFOLDER = "model4-capacity-boundary-154m"

# 4. Upfront Hugging Face Authentication
def resolve_hf_token(token: Optional[str] = None) -> Optional[str]:
    if token and str(token).strip():
        resolved = str(token).strip()
        try:
            login(token=resolved, add_to_git_credential=False)
        except Exception:
            pass
        return resolved

    for env_var in ["HF_TOKEN", "HF", "HUGGINGFACE_TOKEN"]:
        val = os.environ.get(env_var)
        if val and val.strip():
            resolved = val.strip()
            try:
                login(token=resolved, add_to_git_credential=False)
            except Exception:
                pass
            return resolved

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for sec_name in ["HF_TOKEN", "HF", "HUGGINGFACE_TOKEN"]:
            try:
                val = user_secrets.get_secret(sec_name)
                if val and val.strip():
                    resolved = val.strip()
                    try:
                        login(token=resolved, add_to_git_credential=False)
                    except Exception:
                        pass
                    return resolved
            except Exception:
                pass
    except Exception:
        pass

    try:
        hub_tok = get_token()
        if hub_tok and hub_tok.strip():
            return hub_tok.strip()
    except Exception:
        pass

    return None

HF_TOKEN = resolve_hf_token()
if HF_TOKEN:
    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(MODEL_REPO, exist_ok=True, token=HF_TOKEN)
        print(f"Successfully verified write permissions on Hugging Face Hub: '{MODEL_REPO}'")
    except Exception as e:
        print(f"Notice: Upfront HF token verification: {e}")
else:
    print("Notice: HF_TOKEN not found in environment. You can enter it before upload.")


In [ ]:
# Cell 2: Universal Dataset Loader for Phase 4 (Court Judgment Prediction)
@dataclass
class LegalJudgmentDoc:
    doc_id: str
    text: str
    label: Optional[int] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        if self.label is not None:
            self.label = int(self.label)
            if self.label not in (0, 1):
                raise ValueError(f"Document {self.doc_id} has invalid binary label: {self.label}")

def _normalize_dataset_records(raw_data: Any, split_name: str) -> List[LegalJudgmentDoc]:
    docs: List[LegalJudgmentDoc] = []
    if isinstance(raw_data, dict):
        for k, v in raw_data.items():
            if isinstance(v, dict):
                text = v.get("text") or v.get("document") or ""
                lbl = v.get("label") if "label" in v else v.get("labels", v.get("pred", None))
                docs.append(LegalJudgmentDoc(doc_id=str(k), text=str(text), label=lbl, metadata=v))
            elif isinstance(v, str):
                docs.append(LegalJudgmentDoc(doc_id=str(k), text=v, label=None))
    elif isinstance(raw_data, list):
        for idx, item in enumerate(raw_data):
            if isinstance(item, dict):
                doc_id = str(item.get("id") or item.get("doc_id") or f"{split_name}_{idx}")
                text = str(item.get("text") or item.get("document") or "")
                lbl = item.get("label") if "label" in item else item.get("labels", item.get("pred", None))
                docs.append(LegalJudgmentDoc(doc_id=doc_id, text=text, label=lbl, metadata=item))
            else:
                raise ValueError(f"Unrecognized item format at index {idx} in {split_name}")
    else:
        raise ValueError(f"Unrecognized dataset container type: {type(raw_data)} in {split_name}")

    if split_name in ("train", "dev"):
        for doc in docs:
            if doc.label is None:
                raise ValueError(f"FATAL: Missing label in {split_name} document: '{doc.doc_id}'")

    return docs

def load_phase4_splits(repo_id: str = DATASET_REPO, token: Optional[str] = None):
    splits_files = {
        "train": "Phase_4_CJPE/Train/train.json",
        "dev": "Phase_4_CJPE/Train/dev.json",
        "test": "Phase_4_CJPE/Test/test.json",
    }
    loaded = {}
    for key, path in splits_files.items():
        print(f"Downloading Phase 4 {key} split from '{repo_id}/{path}'...")
        local_path = hf_hub_download(repo_id=repo_id, filename=path, repo_type="dataset", token=token)
        with open(local_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        docs = _normalize_dataset_records(raw_data, key)
        loaded[key] = docs
        print(f"  -> Loaded {len(docs):,} {key} documents.")

    train_docs, dev_docs, test_docs = loaded["train"], loaded["dev"], loaded["test"]
    return train_docs, dev_docs, test_docs

train_docs, dev_docs, test_docs = load_phase4_splits(token=HF_TOKEN)
print(f"Dataset Loaded: {len(train_docs):,} Train, {len(dev_docs):,} Dev, {len(test_docs):,} Test")


In [ ]:
# Cell 3: Model 4 Tokenizer & Last K=4 1024-Token Non-Overlapping Chunking Pipeline
def load_model4_tokenizer(repo_id: str = MODEL_REPO, token: Optional[str] = None) -> PreTrainedTokenizerFast:
    tok_path = hf_hub_download(repo_id=repo_id, filename=f"{BACKBONE_SUBFOLDER}/tokenizer.json", token=token)
    tok = PreTrainedTokenizerFast(
        tokenizer_file=tok_path,
        bos_token="<bos>",
        eos_token="<eos>",
        pad_token="<pad>",
        unk_token="<unk>"
    )
    tok.padding_side = "left"
    assert tok.pad_token_id == 3 and tok.eos_token_id == 1, f"Special token ID mismatch: pad={tok.pad_token_id}, eos={tok.eos_token_id}"
    return tok

tokenizer = load_model4_tokenizer(token=HF_TOKEN)

@dataclass
class PreprocessedDocumentBatch:
    doc_id: str
    num_chunks: int
    input_ids: torch.Tensor       # Shape: (M, max_chunk_len)
    attention_mask: torch.Tensor  # Shape: (M, max_chunk_len)
    label: Optional[float] = None

def preprocess_doc_tail_chunks(
    doc: LegalJudgmentDoc,
    tokenizer: PreTrainedTokenizerFast,
    chunk_size: int = 1024,
    k_tail: int = 4
) -> PreprocessedDocumentBatch:
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 3
    tokens = tokenizer.encode(doc.text, add_special_tokens=False)
    if not tokens:
        tokens = [tokenizer.eos_token_id]

    all_chunks = [tokens[i : i + chunk_size] for i in range(0, len(tokens), chunk_size)]
    tail_chunks = all_chunks[-k_tail:]
    M = len(tail_chunks)

    max_chunk_len = max(len(c) for c in tail_chunks)
    padded_ids: List[List[int]] = []
    attn_masks: List[List[int]] = []

    for c in tail_chunks:
        pad_len = max_chunk_len - len(c)
        padded_ids.append([pad_id] * pad_len + c)
        attn_masks.append([0] * pad_len + [1] * len(c))

    input_ids = torch.tensor(padded_ids, dtype=torch.long)
    attention_mask = torch.tensor(attn_masks, dtype=torch.long)
    lbl = float(doc.label) if doc.label is not None else None

    return PreprocessedDocumentBatch(
        doc_id=doc.doc_id,
        num_chunks=M,
        input_ids=input_ids,
        attention_mask=attention_mask,
        label=lbl
    )

class CJPEBatchDataset(Dataset):
    def __init__(self, docs: List[LegalJudgmentDoc], tokenizer: PreTrainedTokenizerFast, k_tail: int = 4):
        self.docs = docs
        self.tokenizer = tokenizer
        self.k_tail = k_tail

    def __len__(self) -> int:
        return len(self.docs)

    def __getitem__(self, idx: int) -> PreprocessedDocumentBatch:
        return preprocess_doc_tail_chunks(self.docs[idx], self.tokenizer, chunk_size=1024, k_tail=self.k_tail)

print("Constructing preprocessed dataset instances for Phase 4...")
train_dataset = CJPEBatchDataset(train_docs, tokenizer, k_tail=4)
dev_dataset = CJPEBatchDataset(dev_docs, tokenizer, k_tail=4)
test_dataset = CJPEBatchDataset(test_docs, tokenizer, k_tail=4)

print(f"Datasets Initialized -> Train: {len(train_dataset):,}, Dev: {len(dev_dataset):,}, Test: {len(test_dataset):,}")


In [ ]:
# Cell 4: Chunk Concat Representation & Learned Chunk Attention Pooling
def extract_chunk_concat_representations(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    Computes Concat(MaskedMeanPool, LastTokenHiddenState) for each chunk in the document:
      h_chunk = [MeanPool(H_valid) ; h_last_token] in R^(2 * d_model)
    """
    M, seq_len, d_model = hidden_states.shape
    chunk_vectors: List[torch.Tensor] = []

    for m in range(M):
        valid_mask = attention_mask[m].unsqueeze(-1).to(hidden_states.dtype)  # (seq_len, 1)
        valid_tokens = hidden_states[m] * valid_mask                         # (seq_len, d_model)
        
        token_count = valid_mask.sum().clamp(min=1.0)
        mean_pool = valid_tokens.sum(dim=0) / token_count                   # (d_model,)
        last_token = hidden_states[m, -1, :]                                # (d_model,)

        chunk_vectors.append(torch.cat([mean_pool, last_token], dim=-1))    # (2 * d_model,)

    return torch.stack(chunk_vectors, dim=0)                                # (M, 2 * d_model)

class ChunkConcatAttentionPooling(nn.Module):
    """
    Learned scalar attention pooling over M chunk representations:
      u_m = tanh(W_a h_m + b_a)
      alpha_m = softmax(v_a^T u_m)
      h_doc = sum(alpha_m * h_m) in R^(2 * d_model)
    """
    def __init__(self, d_model: int = 896, d_attn: int = 128):
        super().__init__()
        self.in_dim = d_model * 2  # 896 * 2 = 1792
        self.W_a = nn.Linear(self.in_dim, d_attn)
        self.v_a = nn.Linear(d_attn, 1, bias=False)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_a.weight)
        nn.init.zeros_(self.W_a.bias)
        nn.init.xavier_uniform_(self.v_a.weight)

    def forward(self, chunk_embeds: torch.Tensor) -> torch.Tensor:
        u = torch.tanh(self.W_a(chunk_embeds))           # (M, d_attn)
        scores = self.v_a(u).squeeze(-1)                 # (M,)
        weights = F.softmax(scores, dim=0)               # (M,)
        doc_embed = torch.sum(weights.unsqueeze(-1) * chunk_embeds, dim=0) # (2 * d_model,)
        return doc_embed

class CourtJudgmentHead(nn.Module):
    """
    2-Layer MLP Classification Head for Court Judgment Prediction:
      LayerNorm(2 * d_model) -> Linear(2 * d_model, hidden_dim) -> GELU -> Dropout(0.1) -> Linear(hidden_dim, 1)
    """
    def __init__(self, d_model: int = 896, hidden_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.in_dim = d_model * 2  # 1792
        self.layer_norm = nn.LayerNorm(self.in_dim)
        self.fc1 = nn.Linear(self.in_dim, hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, doc_embed: torch.Tensor) -> torch.Tensor:
        if doc_embed.dim() == 1:
            doc_embed = doc_embed.unsqueeze(0)
        h = self.layer_norm(doc_embed)
        h = self.dropout(self.act(self.fc1(h)))
        return self.fc2(h)  # (1, 1) or (batch, 1)

attn_pool = ChunkConcatAttentionPooling(d_model=896, d_attn=128).to(device)
head = CourtJudgmentHead(d_model=896, hidden_dim=256, dropout=0.1).to(device)
print(f"Initialized ChunkConcatAttentionPooling and CourtJudgmentHead on {device}.")


In [ ]:
# Cell 5: Model 4 Backbone Loading (100% Full Fine-Tuning Setup)
print(f"Downloading Model 4 configuration and weights from '{MODEL_REPO}/{BACKBONE_SUBFOLDER}'...")
cfg_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/config.json", token=HF_TOKEN)
weights_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/model.safetensors", token=HF_TOKEN)

config = AutoConfig.from_pretrained(cfg_path)
config.use_cache = False
config._name_or_path = MODEL_REPO
base_model = AutoModel.from_config(config)
base_model.config._name_or_path = MODEL_REPO

raw_state_dict = safetensors_load_file(weights_path)
cleaned_sd = {}
for k, v in raw_state_dict.items():
    if k.startswith("model."):
        cleaned_sd[k[len("model."):]] = v
    elif not k.startswith("lm_head."):
        cleaned_sd[k] = v

load_res = base_model.load_state_dict(cleaned_sd, strict=True)
assert len(load_res.missing_keys) == 0, f"Missing keys when loading backbone: {load_res.missing_keys}"
assert len(load_res.unexpected_keys) == 0, f"Unexpected keys when loading backbone: {load_res.unexpected_keys}"
print(f"Base model loaded successfully with strict=True! 100% of Phase 1 weights matched ({len(cleaned_sd)} tensors).")

# Enable 100% Full Fine-Tuning (153.9M parameters unfrozen)
for param in base_model.parameters():
    param.requires_grad = True

base_model = base_model.to(device)

total_backbone = sum(p.numel() for p in base_model.parameters())
trainable_backbone = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
trainable_pool = sum(p.numel() for p in attn_pool.parameters() if p.requires_grad)
trainable_head = sum(p.numel() for p in head.parameters() if p.requires_grad)
total_trainable = trainable_backbone + trainable_pool + trainable_head

print("")
print("="*60)
print(f"Model 4 Total Backbone Parameters:   {total_backbone:,}")
print(f"Trainable Backbone Parameters (100%): {trainable_backbone:,} (100.0%)")
print(f"Trainable Attention Pooling Params:  {trainable_pool:,}")
print(f"Trainable Classification Head Params: {trainable_head:,}")
print(f"Total Trainable System Parameters:    {total_trainable:,}")
print("="*60)
print("")


In [ ]:
# Cell 6: Class Balance Analysis & Loss Objective
train_labels = [int(d.label) for d in train_docs if d.label is not None]
pos_count = sum(train_labels)
neg_count = len(train_labels) - pos_count
imbalance_ratio = neg_count / max(1, pos_count)

pos_weight = torch.tensor([imbalance_ratio], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Train Set Class Balance: {pos_count} Positive (Allowed), {neg_count} Negative (Dismissed)")
print(f"Imbalance Ratio: {imbalance_ratio:.4f} -> BCEWithLogitsLoss pos_weight: {pos_weight.item():.4f}")


In [ ]:
# Cell 7: Differential Optimization Setup & Cosine Schedule
LR_BACKBONE = 5.0e-5
LR_HEAD = 1.0e-3
GRAD_ACCUM_STEPS = 8
EPOCHS = 2

param_groups = [
    {
        "params": [p for p in base_model.parameters() if p.requires_grad],
        "lr": LR_BACKBONE,
        "weight_decay": 0.01,
    },
    {
        "params": list(attn_pool.parameters()) + list(head.parameters()),
        "lr": LR_HEAD,
        "weight_decay": 0.01,
    },
]

optimizer = torch.optim.AdamW(param_groups)
total_steps = math.ceil(len(train_dataset) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(0.10 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print(f"Optimizer: AdamW | Backbone LR: {LR_BACKBONE} | Head/Pool LR: {LR_HEAD} | Steps: {total_steps} (Warmup: {warmup_steps})")


In [ ]:
# Cell 8: Evaluation Function & Optimal Decision Threshold Sweeper
def compute_binary_macro_f1(y_true: List[int], y_pred: List[int]) -> float:
    tp = {0: 0, 1: 0}
    fp = {0: 0, 1: 0}
    fn = {0: 0, 1: 0}
    for gt, pr in zip(y_true, y_pred):
        gt, pr = int(gt), int(pr)
        if gt == pr:
            tp[gt] += 1
        else:
            fp[pr] += 1
            fn[gt] += 1
    f1s = []
    for c in [0, 1]:
        p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
        r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        f1s.append(f1)
    return sum(f1s) / 2.0

def sweep_decision_threshold(y_true: List[int], y_probs: List[float]) -> Tuple[float, float, Dict[float, float]]:
    best_tau = 0.50
    best_f1 = -1.0
    results: Dict[float, float] = {}
    for tau in np.arange(0.10, 0.91, 0.01):
        tau = round(float(tau), 4)
        preds = [1 if p >= tau else 0 for p in y_probs]
        f1 = compute_binary_macro_f1(y_true, preds)
        results[tau] = f1
        if f1 > best_f1:
            best_f1 = f1
            best_tau = tau
    return best_tau, best_f1, results

def evaluate_dev(model, attn_pool, head, dev_dataset, criterion, device) -> Tuple[float, float, float, List[int], List[float]]:
    model.eval()
    attn_pool.eval()
    head.eval()
    total_loss = 0.0
    all_golds: List[int] = []
    all_probs: List[float] = []

    with torch.no_grad():
        for doc_batch in dev_dataset:
            input_ids = doc_batch.input_ids.to(device)
            attention_mask = doc_batch.attention_mask.to(device)
            target = torch.tensor([doc_batch.label], dtype=torch.float32, device=device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
            doc_embed = attn_pool(chunk_reprs)
            logit = head(doc_embed).squeeze(-1)

            loss = criterion(logit, target)
            total_loss += loss.item()

            prob = torch.sigmoid(logit).item()
            all_probs.append(prob)
            all_golds.append(int(doc_batch.label))

    avg_loss = total_loss / max(1, len(all_golds))
    best_tau, best_macro_f1, _ = sweep_decision_threshold(all_golds, all_probs)
    return avg_loss, best_macro_f1, best_tau, all_golds, all_probs


In [ ]:
# Cell 9: 2-Epoch Model 4 Full Fine-Tuning Training Loop
output_dir = "checkpoint_sub01"
os.makedirs(output_dir, exist_ok=True)

best_dev_f1 = 0.0
best_tau = 0.50
history = {"train_loss": [], "dev_loss": [], "dev_macro_f1": []}

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("Commencing 2-epoch Model 4 Full Fine-Tuning Training Loop (Anchor & KD Teacher)...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    base_model.train()
    attn_pool.train()
    head.train()
    running_loss = 0.0
    optimizer.zero_grad()

    # Shuffle training document indices each epoch
    indices = list(range(len(train_dataset)))
    random.shuffle(indices)

    for step, idx in enumerate(indices):
        doc_batch = train_dataset[idx]
        input_ids = doc_batch.input_ids.to(device)
        attention_mask = doc_batch.attention_mask.to(device)
        target = torch.tensor([doc_batch.label], dtype=torch.float32, device=device)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
            chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
            doc_embed = attn_pool(chunk_reprs)
            logit = head(doc_embed).squeeze(-1)
            loss = criterion(logit, target)
            loss_scaled = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_scaled).backward()
        running_loss += loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(indices):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(attn_pool.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        if (step + 1) % 100 == 0 or (step + 1) == len(indices):
            print(f"  Epoch {epoch:02d} | Step {step+1:04d}/{len(indices):04d} | Running Loss: {running_loss / (step + 1):.4f}")

    train_loss = running_loss / len(indices)
    dev_loss, dev_f1, epoch_tau, dev_golds, dev_probs = evaluate_dev(base_model, attn_pool, head, dev_dataset, criterion, device)

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_loss)
    history["dev_macro_f1"].append(dev_f1)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Dev Loss: {dev_loss:.4f} | Dev Macro-F1: {dev_f1:.4f} (Optimal tau*: {epoch_tau:.2f})")

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_tau = epoch_tau
        torch.save(base_model.state_dict(), os.path.join(output_dir, "best_model_weights.pt"))
        torch.save(attn_pool.state_dict(), os.path.join(output_dir, "best_attn_pool.pt"))
        torch.save(head.state_dict(), os.path.join(output_dir, "best_head_weights.pt"))
        np.save(os.path.join(output_dir, "dev_probs.npy"), np.array(dev_probs, dtype=np.float32))
        print(f" -> New Best Dev Macro-F1: {best_dev_f1:.4f}! Checkpoint saved to '{output_dir}'.")

total_train_time = time.time() - start_time
print(f"\nTraining completed in {total_train_time / 60:.2f} minutes. Best Dev Macro-F1: {best_dev_f1:.4f} at tau*={best_tau:.2f}")


In [ ]:
# Cell 10: Teacher Soft Logit Export for Sub 06 (Knowledge Distillation)
print("Loading best Teacher Model checkpoint to export calibrated soft logits...")
base_model.load_state_dict(torch.load(os.path.join(output_dir, "best_model_weights.pt"), map_location=device))
attn_pool.load_state_dict(torch.load(os.path.join(output_dir, "best_attn_pool.pt"), map_location=device))
head.load_state_dict(torch.load(os.path.join(output_dir, "best_head_weights.pt"), map_location=device))

base_model.eval()
attn_pool.eval()
head.eval()

TEMPERATURE = 2.0
teacher_logits_dict: Dict[str, float] = {}

print(f"Computing soft teacher probabilities sigma(z / {TEMPERATURE}) across {len(train_dataset):,} training documents...")
start_export_time = time.time()

with torch.no_grad():
    for i, doc_batch in enumerate(train_dataset):
        input_ids = doc_batch.input_ids.to(device)
        attention_mask = doc_batch.attention_mask.to(device)

        outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
        chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
        doc_embed = attn_pool(chunk_reprs)
        raw_logit = head(doc_embed).squeeze().item()

        # Calibrated soft probability at temperature T=2.0
        soft_prob = torch.sigmoid(torch.tensor(raw_logit / TEMPERATURE)).item()
        teacher_logits_dict[doc_batch.doc_id] = soft_prob

        if (i + 1) % 200 == 0 or (i + 1) == len(train_dataset):
            print(f"  Processed {i+1:04d}/{len(train_dataset):04d} documents...")

export_path = os.path.join(output_dir, "sub01_teacher_logits.pt")
torch.save(teacher_logits_dict, export_path)
print(f"Successfully exported {len(teacher_logits_dict):,} teacher soft targets to '{export_path}' ({os.path.getsize(export_path):,} bytes) in {time.time() - start_export_time:.1f}s.")


In [ ]:
# Cell 11: Final Evaluation, Metrics Report & Loss Plot Visualization
# 1. Plot Training History
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, EPOCHS + 1), history["train_loss"], "o-", label="Train Loss", color="#1f77b4")
ax1.plot(range(1, EPOCHS + 1), history["dev_loss"], "s-", label="Dev Loss", color="#ff7f0e")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("BCE Loss")
ax1.set_title("Phase 4 Sub 01: Model 4 Full FT Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, EPOCHS + 1), history["dev_macro_f1"], "o-", label="Dev Macro-F1", color="#2ca02c")
ax2.axhline(best_dev_f1, color="red", linestyle="--", label=f"Best F1: {best_dev_f1:.4f}")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Macro-F1 Score")
ax2.set_title(f"Phase 4 Sub 01: Dev Macro-F1 (Best: {best_dev_f1:.4f})")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
loss_plot_path = os.path.join(output_dir, "loss_plot.png")
plt.savefig(loss_plot_path, dpi=300)
plt.show()

# 2. Detailed Validation Metrics Report
val_metrics = {
    "submission_id": "sub01",
    "model_backbone": "Model 4 Peak Capacity Boundary Decoder (153.9M)",
    "adaptation_method": "Full Supervised Fine-Tuning (Anchor & KD Teacher)",
    "best_dev_macro_f1": round(float(best_dev_f1), 4),
    "optimal_threshold_tau": round(float(best_tau), 4),
    "epochs": EPOCHS,
    "effective_batch_size": GRAD_ACCUM_STEPS,
    "total_training_time_seconds": round(float(total_train_time), 2),
    "trainable_parameters": total_trainable,
    "total_backbone_parameters": total_backbone,
    "trainable_ratio_percent": 100.0
}

val_metrics_path = os.path.join(output_dir, "val_metrics.json")
with open(val_metrics_path, "w", encoding="utf-8") as f:
    json.dump(val_metrics, f, indent=2)

report_path = os.path.join(output_dir, "training_evaluation_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump({"metrics": val_metrics, "history": history}, f, indent=2)

print(f"Saved evaluation metrics to '{val_metrics_path}' and '{report_path}'.")


In [ ]:
# Cell 12: Test Inference & Strict Codabench Packaging
print("Generating test predictions using optimal threshold tau*...")
base_model.eval()
attn_pool.eval()
head.eval()

test_doc_ids: List[str] = []
test_probs_list: List[float] = []

with torch.no_grad():
    for doc_batch in test_dataset:
        input_ids = doc_batch.input_ids.to(device)
        attention_mask = doc_batch.attention_mask.to(device)

        outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
        chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
        doc_embed = attn_pool(chunk_reprs)
        logit = head(doc_embed).squeeze().item()

        prob = torch.sigmoid(torch.tensor(logit)).item()
        test_probs_list.append(prob)
        test_doc_ids.append(doc_batch.doc_id)

# Save raw test probabilities for ensemble integration
test_probs_arr = np.array(test_probs_list, dtype=np.float32)
np.save(os.path.join(output_dir, "sub01_test_probs.npy"), test_probs_arr)
print(f"Saved test probabilities ({len(test_probs_arr)} cases) to '{os.path.join(output_dir, 'sub01_test_probs.npy')}'.")

# Apply optimal threshold tau* to generate binary predictions
test_binary_preds = [1 if p >= best_tau else 0 for p in test_probs_list]

# Format prediction.json strictly as list-of-dicts [{"id": doc_id, "pred": binary_label}, ...]
prediction_list = [{"id": d_id, "pred": int(p)} for d_id, p in zip(test_doc_ids, test_binary_preds)]

pred_path = os.path.join(output_dir, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(prediction_list, f, indent=2)

# Strict Codabench Packaging: Package prediction.json directly at the zip root
zip_path = os.path.join(output_dir, "submission.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

# Integrity Verification of submission.zip
with zipfile.ZipFile(zip_path, "r") as zf:
    infolist = zf.infolist()
    assert len(infolist) == 1, f"Expected exactly 1 file in submission.zip, found {len(infolist)}"
    assert infolist[0].filename == "prediction.json", f"Found '{infolist[0].filename}' instead of 'prediction.json' at root"

assert len(prediction_list) == len(test_docs), f"Test doc count mismatch: {len(prediction_list)} vs {len(test_docs)}"
for item in prediction_list:
    assert item["pred"] in (0, 1), f"Invalid prediction: {item['pred']}"

print(f"Codabench submission successfully verified and packaged: '{zip_path}' ({os.path.getsize(zip_path):,} bytes, {len(prediction_list)} test cases).")


In [ ]:
# Cell 13: Local Artifact Verification & Atomic Hugging Face Hub Upload (with Executed Notebook Sync)
notebook_filename = "sub01_model4_full_ft.ipynb"

# 1. Save Tokenizer and Model Configuration
tokenizer.save_pretrained(output_dir)
config.save_pretrained(output_dir)

# 2. Stage/Preserve the Executed Notebook File into output_dir
def stage_executed_notebook(nb_name: str, dest_dir: str):
    search_paths = [
        nb_name,
        os.path.join(os.getcwd(), nb_name),
        os.path.join("/kaggle/working", nb_name),
        os.path.join("/kaggle/working", os.path.basename(nb_name)),
    ]
    staged = False
    for p in search_paths:
        if os.path.exists(p) and os.path.isfile(p):
            dest_file = os.path.join(dest_dir, nb_name)
            shutil.copy2(p, dest_file)
            print(f"Successfully staged executed notebook '{p}' -> '{dest_file}' ({os.path.getsize(dest_file):,} bytes).")
            staged = True
            break
    if not staged:
        try:
            import IPython
            IPython.display.display(IPython.display.Javascript('IPython.notebook.save_notebook()'))
            time.sleep(2)
        except Exception:
            pass
        for p in search_paths:
            if os.path.exists(p) and os.path.isfile(p):
                dest_file = os.path.join(dest_dir, nb_name)
                shutil.copy2(p, dest_file)
                print(f"Successfully staged executed notebook '{p}' -> '{dest_file}'.")
                staged = True
                break
    if not staged:
        print(f"Notice: '{nb_name}' not found locally. Ensure notebook is saved before final upload.")

stage_executed_notebook(notebook_filename, output_dir)

# 3. Write Clean YAML README.md for Hugging Face Hub Compliance
readme_path = os.path.join(output_dir, "README.md")
readme_content = f"""---
base_model: {MODEL_REPO}
tags:
- legal
- legal-nlp
- court-judgment-prediction
- ildc
---

# Phase 4 Submission 01: Model 4 Full Fine-Tuning Anchor & KD Teacher
- **Base Model:** `{MODEL_REPO}` (subfolder: `{BACKBONE_SUBFOLDER}`)
- **Task:** Court Judgment Prediction (CJPE - Binary Classification)
- **Role:** Performance Upper-Bound Anchor & Knowledge Distillation Teacher (for Sub 06)
- **Adaptation Method:** Full Parameter Fine-Tuning (100% backbone unfrozen)
- **Best Dev Macro-F1:** {best_dev_f1:.4f} (Optimal Decision Threshold tau*: {best_tau:.2f})
- **Trainable Parameters:** {total_trainable:,} / {total_backbone:,} (100.0%)
- **Teacher Artifacts:** Exported soft targets `sub01_teacher_logits.pt` at temperature T=2.0
"""
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

# 4. Verify All Required Artifacts Exist Locally Before Upload
required_files = [
    "best_model_weights.pt",
    "best_attn_pool.pt",
    "best_head_weights.pt",
    "sub01_teacher_logits.pt",
    "config.json",
    "tokenizer.json",
    "val_metrics.json",
    "training_evaluation_report.json",
    "loss_plot.png",
    "dev_probs.npy",
    "sub01_test_probs.npy",
    "prediction.json",
    "submission.zip",
    "README.md"
]

missing_files = [f for f in required_files if not os.path.exists(os.path.join(output_dir, f))]
assert not missing_files, f"Cannot proceed with upload! Missing local files: {missing_files}"
print(f"All required local artifacts verified present in '{output_dir}'!")

# 5. Atomic Single-Commit Folder Upload to Hugging Face Hub
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter Hugging Face Write Token: ")

api = HfApi(token=HF_TOKEN)
print(f"Initiating atomic upload to '{MODEL_REPO}' under '{TARGET_SUBFOLDER}'...")
api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    path_in_repo=TARGET_SUBFOLDER,
    commit_message=f"Upload Phase 4 Sub 01 (Model 4 Full FT Teacher) - Dev F1: {best_dev_f1:.4f}",
    token=HF_TOKEN
)
print(f"Upload complete! Artifacts & executed notebook published to: https://huggingface.co/{MODEL_REPO}/tree/main/{TARGET_SUBFOLDER}")
